In [19]:
# Importacao de arquivos
import sys
import os
sys.path.append(os.path.abspath('..')) 
%load_ext autoreload
%autoreload 2
from src import *

# Importacao de bibliotecas
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, random_split
import time
import itertools
# Configurações gerais
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'xpu' if hasattr(torch, 'xpu') and torch.xpu.is_available() else 'cpu')
BATCH_SIZE = 16

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Parte 2

In [20]:
def training(model, dataloader, device, peso_fronteira, peso_interior, lr=1e-5, num_epochs=10, use_dice_loss: bool = True):

    pesos = torch.tensor([1.0, peso_interior, peso_fronteira], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=pesos)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    num_batches = len(dataloader)

    print(f"Treinando em: {device}")
    print(f"Batches por época: {num_batches}")

    history = {'loss': [], 'iou': [], 'dice': []}
    for epoch in range(num_epochs):
        model.train()

        accumulated_loss = torch.tensor(0.0, device=device)
        total_intersection = torch.tensor(0.0, device=device)
        total_union = torch.tensor(0.0, device=device)

        for images, masks, _, _ in dataloader:
            images = images.to(device, non_blocking=True, )
            masks = masks.to(device, dtype = torch.long, non_blocking=True)
            optimizer.zero_grad()

            prediction = model(images)
            loss = criterion(prediction, masks)

            if use_dice_loss:
                probs = torch.softmax(prediction, dim=1)
                loss_dice_interior = peso_interior * dice_loss(probs, masks, class_idx=1)
                loss_dice_fronteira = peso_fronteira * dice_loss(probs, masks, class_idx=2)
                loss+= (loss_dice_fronteira+loss_dice_interior)
            
            loss.backward()
            optimizer.step()

            with torch.no_grad(): # validacao
                pred_class = torch.argmax(prediction, dim=1)
                
                # Para acompanhamento durante o treino, calculamos o IoU/Dice 
                # focado apenas na classe "interior" (índice 1)
                interior_pred = (pred_class == 1).float()
                interior_mask = (masks == 1).float()
                
                intersection = (interior_pred * interior_mask).sum()
                union = interior_pred.sum() + interior_mask.sum() - intersection
                
                accumulated_loss += loss.detach()
                total_intersection += intersection
                total_union += union

        epoch_loss = accumulated_loss.item() / num_batches
        ti = total_intersection.item()
        tu = total_union.item()

        epoch_iou = ti / (tu + 1e-6)
        epoch_dice = (2.0 * ti) / (tu + ti + 1e-6)

        history['loss'].append(epoch_loss)
        history['iou'].append(epoch_iou)
        history['dice'].append(epoch_dice)

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | IoU: {epoch_iou:.4f} | Dice: {epoch_dice:.4f}")

    return history


In [21]:
# Consigurações

part = 2

# Dataset
path_train = Path('../data/stage1_train')
path_test = Path('../data/stage1_test')
# full_dataset = DSB2018Dataset(root_dir=path_train, img_size=128, part=part, cache_in_memory=True)
# train_size = int(0.8 * len(full_dataset))
# val_size = len(full_dataset) - train_size
# train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
# test_dataset = DSB2018Dataset(root_dir=path_test, img_size=128, part=part, cache_in_memory=True)

# # DataLoaders
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
# test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

hiperparametros_trilha_a = {
    'espessura_fronteira': [1, 2, 3], # em pixels
    'peso_fronteira': [5.0, 10.0, 20.0], # multiplicador para a classe minoritária
    'peso_interior': [1.0, 1.5]
}

chaves = list(hiperparametros_trilha_a.keys())
combinacoes = list(itertools.product(*hiperparametros_trilha_a.values()))

melhor_map = -1.0
melhor_config = None

print(f"Iniciando busca em grid com {len(combinacoes)} combinações...\n")

for i, valores in enumerate(combinacoes):
    config_atual = dict(zip(chaves, valores))
    
    print(f"[{i+1}/{len(combinacoes)}] Testando: {config_atual}")
    dataset_treino = DSB2018Dataset(root_dir=path_train, img_size=128, part=part, cache_in_memory=True, espessura_fronteira=config_atual['espessura_fronteira'])
    train_loader = DataLoader(dataset_treino, batch_size=BATCH_SIZE, shuffle=True)
    model = UNetTernary()
    model.to(DEVICE)
    
    # 4. Correção: Passar os argumentos de forma nomeada e remover o ":" no final
    history = training(
        model=model, 
        dataloader=train_loader, 
        device=DEVICE, 
        peso_fronteira=config_atual['peso_fronteira'], 
        peso_interior=config_atual['peso_interior'], 
        lr=1e-5, 
        num_epochs=10
    )
    
    # 5. Avaliar o resultado (pegando o Dice da última época como exemplo)
    map_validacao = history['dice'][-1]
    
    if map_validacao > melhor_map:
        melhor_map = map_validacao
        melhor_config = config_atual

print("\nBusca concluída!")
print(f"Melhor pontuação: {melhor_map:.4f}")
print(f"Melhores parâmetros: {melhor_config}")

Iniciando busca em grid com 18 combinações...

[1/18] Testando: {'espessura_fronteira': 1, 'peso_fronteira': 5.0, 'peso_interior': 1.0}
Treinando em: xpu
Batches por época: 42
Epoch 1/10 | Loss: 6.4859 | IoU: 0.0014 | Dice: 0.0028
Epoch 2/10 | Loss: 6.4696 | IoU: 0.0005 | Dice: 0.0010
Epoch 3/10 | Loss: 6.4479 | IoU: 0.0001 | Dice: 0.0003
Epoch 4/10 | Loss: 6.4062 | IoU: 0.0000 | Dice: 0.0000
Epoch 5/10 | Loss: 6.2987 | IoU: 0.0000 | Dice: 0.0000
Epoch 6/10 | Loss: 6.0145 | IoU: 0.0000 | Dice: 0.0000
Epoch 7/10 | Loss: 5.7233 | IoU: 0.0000 | Dice: 0.0000
Epoch 8/10 | Loss: 5.5425 | IoU: 0.0000 | Dice: 0.0000
Epoch 9/10 | Loss: 5.3915 | IoU: 0.0000 | Dice: 0.0000
Epoch 10/10 | Loss: 5.2538 | IoU: 0.0000 | Dice: 0.0000
[2/18] Testando: {'espessura_fronteira': 1, 'peso_fronteira': 5.0, 'peso_interior': 1.5}
Treinando em: xpu
Batches por época: 42
Epoch 1/10 | Loss: 6.9106 | IoU: 0.0000 | Dice: 0.0000
Epoch 2/10 | Loss: 6.8967 | IoU: 0.0000 | Dice: 0.0000
Epoch 3/10 | Loss: 6.8763 | IoU: 0

KeyboardInterrupt: 

In [ ]:
real_mAP, real_count_errors, real_densities, real_samples = evaluate2(model, val_loader, DEVICE)

# Plotando métricas e amostras
plot_metrics(real_mAP, real_count_errors, real_densities)
plot_samples(real_samples, part=part)